In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [2]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [3]:
def transform_smishtank_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [4]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [5]:
smishtank_dataset = pd.read_csv('../Dataset/smishtank.csv', encoding='unicode_escape')
smishtank_dataset.head()

,messageid,Fulltext,Sender,SenderType,timeReceived,MainText,Url,Subdomain,Domain,TLD,...,Phishing,Suspicious,Malware,Brand,URL Subcategory,Message Categories,FullyQualifiedDomain,Domain Registrar,Domain Creation Date,Domain Last Update
0,3,"Text Message\nThu, Jul 29, 19:10\nCostco: Dani...",42003,Short Code,"03/31/2022, 21:58:50","Costco: Daniel, the code 42003 printed on your...",f2gpy.info/RzNKEwsZve,NaN,f2gpy,info,...,0,0,0,Costco,Random Domain,Prize/Contest,f2gpy.info,"NameCheap, Inc.",7/28/2021,7/31/2021
1,5,"<\n+1 (872) 279-0672 >\nText Message\nWed, Feb...",+1 (872) 279-0672,Phone Number,"04/02/2022, 02:59:56","Hi, you still owe UPS $4.10 USD in customs fee...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,UPS,NaN,Delivery,NaN,NaN,NaN,NaN
2,6,"<\n+1 (806) 224-7886 >\nText Message\nThu, Sep...",+1 (806) 224-7886,Phone Number,"04/02/2022, 03:03:00",wel01.us/r/rest05 WELLS FARGO(CS):Profile lock...,wel01.us/r/rest05,NaN,wel01,us,...,0,0,0,WELLS FARGO,Random Domain,Account Alert,wel01.us,"NameCheap, Inc.",8/30/2021,8/30/2021
3,7,Text Message\nToday 2:30 PM\nwho played golf\n...,NaN,NaN,"04/02/2022, 23:42:38","Hi, are you who played golf together last time...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Wrong Number/Romance Scam,NaN,NaN,NaN,NaN
4,8,(8\n+1 (775) 537-4497\ncanador to them\nTato m...,+1 (775) 537-4497,Phone Number,"04/02/2022, 23:46:34","(8 Hi Julianne long time no see, I'm Aleen, ho...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Wrong Number/Romance Scam,NaN,NaN,NaN,NaN


In [6]:
# smishtank_dataset = transform_smishtank_dict(smishtank_dataset)
# smishtank_dataset.head()

In [7]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'smishtank Dataset_'+'.csv')['URL'].to_list())

In [8]:
# smishtank_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'smishtank Dataset_'+'.csv')['URL']
smishtank_dataset['Message Len'] = [len(str(i)) for i in smishtank_dataset['MainText']]
smishtank_dataset.head()

,messageid,Fulltext,Sender,SenderType,timeReceived,MainText,Url,Subdomain,Domain,TLD,...,Suspicious,Malware,Brand,URL Subcategory,Message Categories,FullyQualifiedDomain,Domain Registrar,Domain Creation Date,Domain Last Update,Message Len
0,3,"Text Message\nThu, Jul 29, 19:10\nCostco: Dani...",42003,Short Code,"03/31/2022, 21:58:50","Costco: Daniel, the code 42003 printed on your...",f2gpy.info/RzNKEwsZve,NaN,f2gpy,info,...,0,0,Costco,Random Domain,Prize/Contest,f2gpy.info,"NameCheap, Inc.",7/28/2021,7/31/2021,118
1,5,"<\n+1 (872) 279-0672 >\nText Message\nWed, Feb...",+1 (872) 279-0672,Phone Number,"04/02/2022, 02:59:56","Hi, you still owe UPS $4.10 USD in customs fee...",NaN,NaN,NaN,NaN,...,NaN,NaN,UPS,NaN,Delivery,NaN,NaN,NaN,NaN,128
2,6,"<\n+1 (806) 224-7886 >\nText Message\nThu, Sep...",+1 (806) 224-7886,Phone Number,"04/02/2022, 03:03:00",wel01.us/r/rest05 WELLS FARGO(CS):Profile lock...,wel01.us/r/rest05,NaN,wel01,us,...,0,0,WELLS FARGO,Random Domain,Account Alert,wel01.us,"NameCheap, Inc.",8/30/2021,8/30/2021,120
3,7,Text Message\nToday 2:30 PM\nwho played golf\n...,NaN,NaN,"04/02/2022, 23:42:38","Hi, are you who played golf together last time...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Wrong Number/Romance Scam,NaN,NaN,NaN,NaN,56
4,8,(8\n+1 (775) 537-4497\ncanador to them\nTato m...,+1 (775) 537-4497,Phone Number,"04/02/2022, 23:46:34","(8 Hi Julianne long time no see, I'm Aleen, ho...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Wrong Number/Romance Scam,NaN,NaN,NaN,NaN,62


In [9]:
smishtank_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'Smishtank Websites Analysis'+'.csv')
# smishtank_website_analysis_data = smishtank_website_analysis_data.drop(columns=['ham', 'spam'])
smishtank_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,83hm2uaj.kzra.in/?/OVA013,83hm2uaj.kzra.in,0,0,-1,0
1,https://usps-intend.shop,usps-intend.shop,0,0,-1,0
2,https://aritc.yru.ac.th/redirect/78?url=bit.do...,aritc.yru.ac.th,0,0,200,0
3,http://site.rdv360.com/free/main45/mail?=924514,site.rdv360.com,14187,89,200,0
4,https://urlsr.com/LY9llxL,urlsr.com,310,12,200,0


In [10]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [11]:
smishtank_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_24448\3839705074.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  smishtank_website_analysis_data.iloc[0][0]


'83hm2uaj.kzra.in/?/OVA013'

In [12]:
# for row in smishtank_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [13]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [14]:
extracted_urls = smishtank_dataset['Url'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(smishtank_website_analysis_data['FQDN'])}
website_data = smishtank_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [15]:
smishtank_dataset['FQDN'] = fqdn
smishtank_dataset['Website Size in KB'] = website_size
smishtank_dataset['Website Textual Content Length'] = text_content_len
smishtank_dataset['Status Code'] = status_code
smishtank_dataset['Parked'] = parked

In [16]:
smishtank_dataset = smishtank_dataset.replace('', np.nan)
smishtank_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_24448\3243891224.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  smishtank_dataset = smishtank_dataset.replace('', np.nan)


,messageid,Fulltext,Sender,SenderType,timeReceived,MainText,Url,Subdomain,Domain,TLD,...,FullyQualifiedDomain,Domain Registrar,Domain Creation Date,Domain Last Update,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,3,"Text Message\nThu, Jul 29, 19:10\nCostco: Dani...",42003,Short Code,"03/31/2022, 21:58:50","Costco: Daniel, the code 42003 printed on your...",f2gpy.info/RzNKEwsZve,NaN,f2gpy,info,...,f2gpy.info,"NameCheap, Inc.",7/28/2021,7/31/2021,118,f2gpy.info,0.0,0.0,-1.0,0.0
1,5,"<\n+1 (872) 279-0672 >\nText Message\nWed, Feb...",+1 (872) 279-0672,Phone Number,"04/02/2022, 02:59:56","Hi, you still owe UPS $4.10 USD in customs fee...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,128,NaN,NaN,NaN,NaN,NaN
2,6,"<\n+1 (806) 224-7886 >\nText Message\nThu, Sep...",+1 (806) 224-7886,Phone Number,"04/02/2022, 03:03:00",wel01.us/r/rest05 WELLS FARGO(CS):Profile lock...,wel01.us/r/rest05,NaN,wel01,us,...,wel01.us,"NameCheap, Inc.",8/30/2021,8/30/2021,120,wel01.us,0.0,0.0,-1.0,0.0
3,7,Text Message\nToday 2:30 PM\nwho played golf\n...,NaN,NaN,"04/02/2022, 23:42:38","Hi, are you who played golf together last time...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,56,NaN,NaN,NaN,NaN,NaN
4,8,(8\n+1 (775) 537-4497\ncanador to them\nTato m...,+1 (775) 537-4497,Phone Number,"04/02/2022, 23:46:34","(8 Hi Julianne long time no see, I'm Aleen, ho...",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,62,NaN,NaN,NaN,NaN,NaN


In [17]:
# Counter(smishtank_dataset['FQDN'].to_list())
smishtank_dataset[(smishtank_dataset['Url'].notna()) & (smishtank_dataset['FQDN'].isna())]

,messageid,Fulltext,Sender,SenderType,timeReceived,MainText,Url,Subdomain,Domain,TLD,...,FullyQualifiedDomain,Domain Registrar,Domain Creation Date,Domain Last Update,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [18]:
# Counter(smishtank_dataset['FQDN'].to_list())
smishtank_dataset[(smishtank_dataset['Url'].notna()) & (smishtank_dataset['FQDN'].notna())]

,messageid,Fulltext,Sender,SenderType,timeReceived,MainText,Url,Subdomain,Domain,TLD,...,FullyQualifiedDomain,Domain Registrar,Domain Creation Date,Domain Last Update,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,3,"Text Message\nThu, Jul 29, 19:10\nCostco: Dani...",42003,Short Code,"03/31/2022, 21:58:50","Costco: Daniel, the code 42003 printed on your...",f2gpy.info/RzNKEwsZve,NaN,f2gpy,info,...,f2gpy.info,"NameCheap, Inc.",7/28/2021,7/31/2021,118,f2gpy.info,0.0,0.0,-1.0,0.0
2,6,"<\n+1 (806) 224-7886 >\nText Message\nThu, Sep...",+1 (806) 224-7886,Phone Number,"04/02/2022, 03:03:00",wel01.us/r/rest05 WELLS FARGO(CS):Profile lock...,wel01.us/r/rest05,NaN,wel01,us,...,wel01.us,"NameCheap, Inc.",8/30/2021,8/30/2021,120,wel01.us,0.0,0.0,-1.0,0.0
6,11,"<3\nHC\n+1 (859) 398-4745\n+\nMon, Dec 13\nDea...",+1 (859) 398-4745,Phone Number,"04/03/2022, 00:16:34",Dear. You are invited to join the (Bitcoin) in...,https://chat.whatsapp.com/Djci PB8b7gTGGt16QJ5mJm,chat,whatsapp,com,...,chat.whatsapp.com,"RegistrarSafe, LLC",9/4/2008,NaN,245,chat.whatsapp.com,181682.0,3040.0,200.0,0.0
7,12,<\n+1 (626) 628-7248 >\nO\nText Message\nMonda...,+1 (626) 628-7248,Phone Number,"04/03/2022, 00:43:01","CITI: Unauthorized activity was detected, we h...",http://citibsec.com,NaN,citibsec,com,...,citibsec.com,Registermax,5/3/2022,5/3/2022,146,citibsec.com,0.0,0.0,-1.0,0.0
10,15,"9:11\n? ?\n???\n? (803) 307-... ?\nSunday, Jan...",(803) 307-,Phone Number,"04/03/2022, 02:12:57","BofA:Your account has been restricted, visit s...",secbofa.page.link/NLt,secbofa,page,link,...,secbofa.page.link,"MarkMonitor, Inc.",2/9/2017,NaN,66,secbofa.page.link,0.0,0.0,400.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1083,1806,4:35\n\nwaters.javier@hotmail.com>\niMessage\...,waters.javier@hotmail.com,Email To Text,"12/11/2023, 00:35:14",U.S.P.S - Your package arrived at the warehous...,https://uspxzpl.top,NaN,uspxzpl,top,...,uspxzpl.top,"NameSilo, LLC",10/5/2023,12/8/2023,322,uspxzpl.top,0.0,0.0,-1.0,0.0
1084,1807,"4:35\n+\n\n+66 82-946-0849 >\niMessage\nThu, ...",+66 82-946-0849,Phone Number,"12/11/2023, 00:36:08",USPS service: Package was delayed due to incom...,https://www.upotps.com/go/Tracking/G01U6,www,upotps,com,...,www.upotps.com,ALIBABA.COM,11/24/2022,11/25/2023,196,www.upotps.com,1170.0,0.0,200.0,1.0
1088,1816,..ll Verizon\n10\n9:42 AM\nfrosssusanberry@gma...,frosssusanberry@gmail.com,Email To Text,"12/12/2023, 14:34:27",Hello dear user Your package has arrived at th...,https://usrpxs.top,NaN,usrpxs,top,...,usrpxs.top,"NameSilo, LLC",12/6/2023,12/7/2023,400,usrpxs.top,0.0,0.0,-1.0,0.0
1089,1817,+1 (769) 296-5436 >\nText Message\nYesterday 1...,+1 (769) 296-5436,Phone Number,"12/13/2023, 15:01:23","Hey, it's John here, I'm having a hard time de...",zskuyyd.com/aj4BdNxG,NaN,zskuyyd,com,...,zskuyyd.com,"NameCheap, Inc.",12/11/2023,1/1/2001,129,zskuyyd.com,1038.0,0.0,200.0,1.0


In [19]:
print(len(smishtank_dataset))

1091


In [21]:
#messages with URL
print(len(smishtank_dataset[(smishtank_dataset['Url'].notna())]), len(smishtank_dataset[(smishtank_dataset['Url'].notna())])/len(smishtank_dataset))

938 0.8597616865261228


In [ ]:
# #smish messages with URL
# len(smishtank_dataset[(smishtank_dataset['Extracted URL'].notna()) & (smishtank_dataset['class']==1)])

In [ ]:
# #smish messages with URL
# len(smishtank_dataset[(smishtank_dataset['Extracted URL'].notna()) & (smishtank_dataset['class']==0)])

In [22]:
#unique FQDN
len(set(smishtank_dataset[(smishtank_dataset['FQDN'].notna())]['FQDN']))

740

In [23]:
only_unique_live_websites_data = smishtank_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

101


In [24]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

32


In [ ]:
smishtank_dataset.to_csv('../Dataset/Refined_Smishtank_Dataset.csv', index=None)